# 03. 세계 모델로 계획하기와 simulator 편향

## 목표
별도 세계 모델이 후보 행동의 다음 상태를 예측해 정책 계획을 돕는 방식을 구현합니다. 부정확한 simulator가 정책을 잘못 이끄는 문제도 실험합니다.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Prediction:
    next_state: str
    reward: float

# 실제 환경: 지름길에는 함정이 있고 안전 경로는 두 단계가 필요합니다.
real_dynamics = {
    ("start", "shortcut"): Prediction("trap", -5.0),
    ("start", "safe"): Prediction("checkpoint", 0.0),
    ("checkpoint", "finish"): Prediction("goal", 10.0),
}

# 편향된 simulator는 지름길의 실패를 관찰하지 못해 높은 보상을 예측합니다.
biased_model = dict(real_dynamics)
biased_model[("start", "shortcut")] = Prediction("goal", 12.0)

actions = {"start": ["shortcut", "safe"], "checkpoint": ["finish"]}

In [ ]:
def plan_one_step(state, model):
    candidates = []
    for action in actions.get(state, []):
        prediction = model[(state, action)]
        candidates.append((prediction.reward, action, prediction.next_state))
    return max(candidates)

for name, model in [("정확한 세계 모델", real_dynamics), ("편향된 세계 모델", biased_model)]:
    reward, action, predicted_state = plan_one_step("start", model)
    actual = real_dynamics[("start", action)]
    print(name)
    print(f"  선택={action}, 예측=({predicted_state}, {reward}), 실제=({actual.next_state}, {actual.reward})")

## 해석

세계 모델을 정책 최적화에 사용하면 정책은 simulator의 오류까지 최적화합니다. 실제 환경에서 드물게 발생하는 API 오류, 권한 거부, timeout을 simulator가 누락하면 에이전트는 현실에서 취약한 행동을 선호할 수 있습니다. Qwen-AgentWorld의 제어 지시처럼 부분 실패를 의도적으로 생성하고, 실제 환경의 held-out 전이로 fidelity를 평가해야 합니다.

## 확장 과제

1. 두 단계 lookahead를 구현해 안전 경로의 최종 보상 10을 발견하게 하세요.
2. 예측 불확실성이 큰 행동에 penalty를 적용하세요.
3. 실제 환경 표본을 추가할 때 편향된 전이 확률을 갱신하는 tabular 학습기를 만드세요.
4. simulator에서 고른 행동을 실제 환경에서 일정 비율 검증하는 혼합 rollout 정책을 설계하세요.